# Visor de P0 sistemático: qué vio el sistema en cada análisis

**Para qué sirve.** Elegís un análisis por su short-id, los primeros 8 caracteres del id. El cuaderno dibuja, para
cada temporalidad (1D, 12H, 4H y 1H), lo que el motor de P0 sistemático vio en el momento del análisis:
- **un gráfico de fractales**, con los techos y pisos, el zigzag, el tramo A→B y la línea de invalidación;
- **un gráfico de Fibonacci**, con los niveles del tramo, sus zonas, los eventos (por dónde entró el precio y si rechazó
  o rompió) y tus niveles del análisis.

Así podés comparar, TF por TF, lo que leyó el sistema con lo que escribiste en tu tesis de P0.

**Cómo usarlo.**
1. En la celda de configuración (sección 2), cambiá `SYMBOL` (`"XAUUSD"` o `"BTCUSD"`) y `SHORT_ID`.
2. Ejecutá todo el cuaderno (Run All).
3. **Opción 3, si querés corregir al sistema:**
   - en cada gráfico de fractales, los últimos 8 pivotes llevan una letra (a = el más viejo);
   - escribí en `ELECCION` qué letras son tu A y tu B en esa TF (y, si querés, su estado en `ESTADO`). Para un
     pivote más viejo que las 8 letras, usá la fecha de su vela, que sale en la lista de `LISTAR_PIVOTES`;
   - volvé a ejecutar. El cuaderno recalcula niveles, eventos, puntajes y el P0 de cada modelo, y los muestra al lado
     de lo que había propuesto el sistema.

**Qué no hace.** No escribe nada. Lee la DB de la cuenta en solo lectura y las velas de la foto fija de la
evaluación o, si lo pedís, del banco vivo. Usa los mismos motores que la evaluación (`tools/p0_sistematico.py`).

## 1. Cómo funciona el sistema, explicado simple

**Las velas.** Cada vela resume lo que hizo el precio en un rato: en 1H, una hora; en 1D, un día. La mecha de arriba es
el precio más alto de ese rato y la de abajo, el más bajo. En estos gráficos, una vela azul cerró más arriba de donde
abrió y una roja, más abajo.

**1. Los fractales: montañas y valles.** El sistema recorre las velas mirando las alturas.
- Una vela es un **techo** (una montaña) si su máximo es más alto que el de las 2 velas de su izquierda y que el de las
  2 de su derecha.
- Un **piso** (un valle) es lo mismo al revés, con los mínimos.
- **Hay que esperar:** para saber que una vela es una montaña hacen falta las 2 velas de después. Por eso el sistema
  recién "ve" un techo cuando cerró la segunda vela posterior. En los gráficos, esa espera es la línea punteada que va
  del fractal a la vela en que se confirmó. Gracias a esa espera, el sistema nunca usa información del futuro.
- **Empates:** si el máximo empata con el de una vela anterior, igual puede ser montaña. Si empata con una posterior,
  no, porque después vino otra igual de alta.

**2. El zigzag y la escalera.** El sistema une montañas y valles alternados: montaña, valle, montaña, valle.
- Si aparecen dos montañas seguidas, sin un valle en el medio, se queda con la más alta. La otra queda "descartada" y
  se dibuja hueca, en gris. Con dos valles seguidos, se queda con el más bajo.
- Compara cada montaña con la anterior: si es más alta es **HH**, y si es más baja, **LH**. Compara cada valle con el
  anterior: más alto es **HL**, más bajo es **LL**.
- **Escalera que sube** (último techo HH y último piso HL): **alcista**. **Escalera que baja** (LH y LL): **bajista**.
  Cualquier mezcla: **rango**.
- **Si se rompe la escalera:** si estaba alcista y una vela **cierra** debajo del último valle (el último HL), pasa a
  rango. Vuelve a evaluarse cuando aparecen una montaña y un valle nuevos.

**3. El tramo A→B.** Es el último salto de la escalera:
- alcista: A es el valle desde donde subió y B, la montaña donde terminó;
- bajista: A es la montaña desde donde bajó y B, el valle donde terminó;
- rango: el último salto, para cualquier lado.

**4. Fibonacci: una regla sobre el salto.** Imaginá una regla puesta sobre el tramo, con el 0 en B (la punta) y el 1
en A (el arranque). Los niveles de P0 son marcas en esa regla: 0.382, 0.5 y 0.618 dicen cuánto devolvió el precio del
salto.
- **Ejemplo:** A = 4000 y B = 4100, o sea que subió 100. El 0.382 queda 38.2 puntos debajo de B, en 4061.8; el 0.5,
  en 4050; el 0.618, en 4038.2. La fórmula es nivel = B − r · (B − A).
- **La zona:** cada nivel es una franja de ±2.5% del salto. En el ejemplo son ±2.5 puntos: de 4059.3 a 4064.3 para el
  0.382.

**5. El evento: qué hizo el precio al llegar al nivel.**
- **Entrada:** la primera vez, después de confirmado B, que el precio toca la zona viniendo de afuera; la mecha cuenta.
  El sistema anota de qué lado llegó: "↓" si venía bajando, "↑" si venía subiendo.
- **Dos líneas de llegada**, a ±10% del salto desde el nivel. En el ejemplo son ±10 puntos:
  - si el precio se aleja 10 puntos hacia el lado contrario al que venía, es un **rechazo**: rebotó. En el ejemplo,
    vuelve a 4071.8;
  - si sigue 10 puntos para el mismo lado, es una **ruptura**: cruzó. En el ejemplo, llega a 4051.8;
  - si una misma vela toca las dos líneas, no se sabe cuál tocó primero: es **ambiguo** y no cuenta.
- **Qué velas usa:** las más finas que hay (de 1 minuto, si existen) para ver cuál línea se tocó primero. En fechas
  viejas, sin velas finas, usa las de la temporalidad del tramo.
- **Tramo gastado:** si el precio pasa B (hace un máximo nuevo), el tramo ya no anota eventos nuevos.

**6. Cada temporalidad por separado.** 1D, 12H, 4H y 1H tienen cada una su zigzag, su tramo, sus niveles y sus eventos.
Una TF puede estar alcista y otra en rango al mismo tiempo.

**7. El puntaje de cada TF.** Se calcula en el momento del análisis (el ancla), mirando solo velas ya cerradas.
- En rango, o sin historia suficiente: 0.
- Modelo A (solo estructura): +1 si la TF está alcista, −1 si está bajista.
- Modelos B a H (con Fibonacci), en una TF alcista; en una bajista es lo mismo con signo negativo:

  | El último evento del tramo fue… | Puntaje | Idea |
  |---|---|---|
  | rechazo en un retroceso (0.382, 0.5, 0.618) o en 1 | +2 | devolvió y rebotó: la tendencia sigue firme |
  | ruptura de un retroceso o de 1 | 0 | devolvió de más: dudoso |
  | rechazo en 0 (el precio volvió a B y rebotó para abajo) | 0 | doble techo: dudoso |
  | ruptura de 0 (pasó B) | +2 | rompió el máximo |
  | todavía ningún evento con resultado | +1 | solo la tendencia |

**8. El P0 del modelo.** Suma los puntajes con los pesos del modelo y los pasa a la escala de tu P0 (−2 a +2). El
signo es la dirección (+ Long, − Short), y el tamaño, la fuerza (1 = Mid, 2 = Strong). 0 quiere decir que no opina.
- **Ejemplo con B** (pesos 0.25 en cada TF): 1D +2, 12H +1, 4H 0 y 1H −1.
- Suma: 0.25 · (2 + 1 + 0 − 1) = 0.5.
- Como B puede llegar a 2, se divide por 2 y se multiplica por 2: queda 0.5, que redondea a +1. Resultado: **Long Mid**.

**9. Los gemelos placebo.** Bp a Hp hacen exactamente lo mismo que su modelo, pero con marcas falsas en la regla:
0.10, 0.300, 0.441, 0.559 y 0.90. Son como la pastilla de azúcar de un experimento: si el modelo con Fibonacci no le
gana al que usa marcas falsas, lo que ayuda no es Fibonacci, sino mirar rebotes en cualquier parte del retroceso.

Los dos gráficos siguientes muestran todo esto sobre una serie inventada de 18 velas. Hay un tramo alcista de 10 a 20;
el precio baja, entra a la zona del 0.382 y rebota.

In [ ]:
import os, sys
from datetime import datetime, timedelta

REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "jupyter" else os.getcwd()
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from core.candle_resolution import Candle
from tools import p0_sistematico as ps
from tools import p0_visor as pv

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

# Ejemplo inventado: (máximo, mínimo, cierre) por vela de 1H.
_T0 = datetime(2026, 6, 1)
_rows = [(10.0, 9.0, 9.5), (11.0, 9.8, 10.5), (12.0, 11.0, 11.5), (11.5, 10.0, 10.2), (11.0, 9.2, 9.6),
         (12.5, 10.0, 12.0), (13.5, 11.5, 13.0), (13.0, 11.0, 11.5), (12.0, 10.0, 10.5), (13.0, 10.5, 12.8),
         (15.0, 12.5, 14.8), (20.0, 14.5, 19.5), (19.8, 18.0, 18.5), (18.6, 16.5, 17.0), (17.2, 16.0, 16.2),
         (17.3, 16.1, 17.2), (18.0, 17.0, 17.8), (18.5, 17.5, 18.2)]
_ejemplo = ps.Market("EJEMPLO", {"1H": [Candle(_T0 + i * timedelta(hours=1), (h + l) / 2, h, l, c)
                                         for i, (h, l, c) in enumerate(_rows)]})
_v = pv.tf_view(_ejemplo, "1H", _T0 + timedelta(hours=16), window=30, future=2)
fig, axes = plt.subplots(2, 1, figsize=(13, 9))
pv.plot_fractals(axes[0], _v)
pv.plot_fibonacci(axes[1], _v)
fig.tight_layout()
plt.show()
display(Markdown(pv.explain_tf(_v)))

## 2. Configuración

Cambiá estos valores y ejecutá todo el cuaderno.
- `SYMBOL`: `"XAUUSD"` o `"BTCUSD"`.
- `SHORT_ID`: los primeros caracteres del id del análisis; alcanza con que identifiquen uno solo.
- `TFS`: las temporalidades a dibujar. Podés agregar `"1W"`, que usa el modelo H.
- `VENTANA`: cuántas velas antes del ancla mostrar. Si el tramo A→B empieza antes, se amplía hasta verlo entero.
- `FUTURO`: cuántas velas después del ancla mostrar, sombreadas: el modelo no las vio. Por defecto 0.
- `MOSTRAR_RESULTADO`: si mostrar la sección 6 (qué pasó después). Por defecto no.
- `MOSTRAR_PLACEBO`: dibujar o no los niveles placebo.
- `FUENTE_VELAS`:
  - `"foto"`: la foto fija de la evaluación (hasta el 2026-10-05/06). Es la que usan los resultados del informe;
  - `"banco"`: el banco vivo `.data/candle_bank`, en solo lectura, para análisis posteriores a la foto.
- `ELECCION` (opción 3): tu tramo A→B por TF, con las letras del gráfico de fractales. Por ejemplo,
  `{"4H": ("c", "f")}` quiere decir A = c y B = f en 4H. `None` deja el tramo que propone el sistema.
  - Un pivote más viejo que las 8 letras se elige por la fecha de su vela: `{"12H": ("2026-06-17 03:00", "a")}`. En
    1D alcanza con el día (`"2026-03-09"`). Si esa vela no es un pivote confirmado, el error lista los más cercanos.
- `LISTAR_PIVOTES`: cuántos pivotes listar por TF, con su letra y su fecha, para elegirlos en `ELECCION` (0 = ninguno).
- **Elegí tus tramos sin ver el futuro:** con `FUTURO = 0` y `MOSTRAR_RESULTADO = False` (los valores por defecto)
  tu elección usa lo mismo que tenía el sistema en el ancla. Cambialos solo después de haber elegido.
- `ESTADO` (opcional): el estado de una TF elegida, `"alcista"`, `"bajista"` o `"rango"`. Si no lo das, sale de la
  dirección de tu tramo.
- `RUPTURA`: a qué distancia del nivel cuenta una ruptura, en fracción del tramo. 0.10 es la regla de la evaluación; el
  rechazo queda siempre en 0.10.

In [ ]:
SYMBOL = "XAUUSD"          # "XAUUSD" o "BTCUSD"
SHORT_ID = "21e49870"      # los primeros caracteres del id del análisis
TFS = ("1D", "12H", "4H", "1H")
VENTANA = 90               # velas antes del ancla (se amplía si hace falta para ver A)
FUTURO = 0                 # velas después del ancla, sombreadas (0 = no mirar el futuro al elegir)
MOSTRAR_RESULTADO = False  # sección 6: qué pasó después
MOSTRAR_PLACEBO = True
FUENTE_VELAS = "foto"      # "foto" o "banco"
ELECCION = {"1D": None, "12H": None, "4H": None, "1H": None}   # p. ej. {"4H": ("c", "f")} o una fecha
LISTAR_PIVOTES = 0         # p. ej. 16: lista los últimos 16 pivotes de cada TF con su fecha
ESTADO = {}                # p. ej. {"4H": "bajista"}
RUPTURA = 0.10             # 0.10 = regla de la evaluación; 0.15, 0.20, 0.25 alejan solo la ruptura

## 3. El análisis

In [ ]:
MAIN_CHECKOUT = os.environ.get("P0_MAIN_CHECKOUT", "/home/jorgecg/projects/trading/blast_master")
DB = os.path.join(MAIN_CHECKOUT, ".data", ps.SYMBOLS[SYMBOL])
if FUENTE_VELAS == "foto":
    VELAS = os.path.join(REPO, ".tmp", "p0_foto_20261006")
    ps.verify_snapshot(VELAS, os.path.join(REPO, "jupyter", "p0_sistematico", "foto_manifest.json"))
elif FUENTE_VELAS == "banco":
    VELAS = os.path.join(MAIN_CHECKOUT, ".data", "candle_bank")
else:
    raise ValueError("FUENTE_VELAS tiene que ser 'foto' o 'banco'")
MARKET = ps.load_market(VELAS, SYMBOL, rupture_fraction=RUPTURA)
INFO = pv.find_analysis(DB, SYMBOL, SHORT_ID)
SP, TESIS = pv.thesis_direction(INFO, MARKET)
last_1h = MARKET.bars["1H"].times[-1]
if INFO.anchor > last_1h:
    print(f"OJO: el ancla ({INFO.anchor}) es posterior a la última vela de 1H de la fuente ({last_1h}). "
          "Usá FUENTE_VELAS = 'banco'.")
ficha = {
    "id": INFO.trade_id, "creado (Confirm & Save)": INFO.created_at, "ancla (el sistema mira hasta acá)": INFO.anchor,
    "retroactivo": ("sí, recuperado (cuenta)" if INFO.recovered else "sí (fuera de la evaluación)")
    if INFO.is_backdated else "no",
    "sesgo del edge": INFO.bias, "tu P0": f"{INFO.operator_direction} {INFO.operator_strength or ''}".strip(),
    "tu objetivo (EVP)": INFO.evp, "tu invalidación (SI)": INFO.si, "precio de partida": SP,
    "dirección de la tesis": TESIS,
}
display(pd.DataFrame({"campo": list(ficha), "valor": [str(v) for v in ficha.values()]}))
display(Markdown("**Tu tesis de P0:**\n\n" + "\n".join("> " + line for line in (INFO.thesis or "(vacía)").splitlines())))

## 4. Qué dijo el sistema y qué dijiste vos

La primera tabla compara, por TF, tu etiqueta de la tesis (`T-BL` = tramo alcista, `T-BR` = bajista) con el tramo que
tomó el sistema y, si usaste `ELECCION`, con el tuyo. La segunda tiene el P0 de cada modelo en el ancla: con lo que
propuso el sistema y, al lado, con tus tramos.

In [ ]:
VIEWS = {tf: pv.tf_view(MARKET, tf, INFO.anchor, window=VENTANA, future=FUTURO, choice=ELECCION.get(tf),
                        state_choice=ESTADO.get(tf)) for tf in TFS}


def _leg_txt(leg, view):
    if leg is None:
        return "—"
    return f"{pv.pivot_name(view, leg.a)}→{pv.pivot_name(view, leg.b)} {'sube (BL)' if leg.up else 'baja (BR)'}"


rows = []
for tf, v in VIEWS.items():
    row = {"TF": tf, "tu etiqueta": INFO.labels.get(tf, "—"),
           "tu ==": f"== {INFO.refs[tf]}" if tf in INFO.refs else "",
           "tramo del sistema": _leg_txt(v.system_leg, v), "estado del sistema": pv.STATE_ES[v.system_state],
           "puntaje sistema (A / B–H)": f"{v.system_scores[0]:+d} / {v.system_scores[1]:+d}"}
    if v.chosen:
        row.update({"tu tramo": _leg_txt(v.leg, v), "tu estado": pv.STATE_ES[v.state],
                    "puntaje tuyo (A / B–H)": f"{v.score_structure:+d} / {v.score_fib:+d}"})
    rows.append(row)
display(pd.DataFrame(rows).fillna(""))

SUMMARY_SYS = pv.model_summary(MARKET, INFO.anchor)
SUMMARY = pv.model_summary(MARKET, INFO.anchor, VIEWS)
CHOSE = any(v.chosen for v in VIEWS.values())
if CHOSE:
    both = SUMMARY_SYS[["modelo", "descripción", "P0 (x0)", "dirección", "fuerza"]].rename(
        columns={"P0 (x0)": "x0 sistema", "dirección": "dirección sistema", "fuerza": "fuerza sistema"})
    both["x0 con tu elección"] = SUMMARY["P0 (x0)"]
    both["dirección con tu elección"] = SUMMARY["dirección"]
    both["fuerza con tu elección"] = SUMMARY["fuerza"]
    display(both)
else:
    display(SUMMARY_SYS)

## 5. TF por TF: fractales y Fibonacci

Para cada temporalidad:
- la explicación en palabras de lo que leyó el sistema;
- el gráfico de fractales, con el zigzag, el tramo A→B y la invalidación;
- el gráfico de Fibonacci, con las zonas de P0, los placebo, los eventos, las barreras y tus niveles.

La línea vertical es el ancla. A la derecha, lo sombreado es el futuro, que el modelo no vio.

In [ ]:
for tf, v in VIEWS.items():
    display(Markdown(f"### {tf}"))
    display(Markdown(pv.explain_tf(v)))
    if LISTAR_PIVOTES:
        display(pv.list_pivots(MARKET, tf, INFO.anchor, LISTAR_PIVOTES))
    fig, ax = plt.subplots(figsize=(14, 5.2))
    pv.plot_fractals(ax, v, INFO)
    fig.tight_layout()
    plt.show()
    fig, ax = plt.subplots(figsize=(14, 5.6))
    pv.plot_fibonacci(ax, v, INFO, show_placebo=MOSTRAR_PLACEBO)
    fig.tight_layout()
    plt.show()

## 6. Qué pasó después

Esto el modelo no lo vio. Sirve para mirar, no para ajustar nada.
- **Qué nivel tocó primero el precio:** sale del resolvedor de la spec 002, desde el ancla.
- **Si cada P0 acertó:** Long acierta si el precio toca primero el nivel de arriba; Short, si toca primero el de
  abajo.
  - S1: el primer toque, sin límite de tiempo.
  - S4 estricto: solo cuenta si toca dentro de 48 h.

In [ ]:
preds = {"tu P0": 1 if INFO.operator_direction == "Long" else -1 if INFO.operator_direction == "Short" else 0}
for table, tag in ((SUMMARY_SYS, "sistema"), (SUMMARY, "con tu elección")):
    if tag == "con tu elección" and not CHOSE:
        continue
    for _, row in table.iterrows():
        if row.modelo in ("A", "B", "C", "D", "E", "F", "G", "H"):
            preds[f"modelo {row.modelo} ({tag})"] = 1 if row["P0 (x0)"] > 0 else -1 if row["P0 (x0)"] < 0 else 0
if MOSTRAR_RESULTADO:
    out = pv.what_happened(INFO, MARKET, preds)
    display(pd.DataFrame({"": list(out), "valor": [str(v) for v in out.values()]}))
else:
    print("Oculto: poné MOSTRAR_RESULTADO = True después de haber elegido tus tramos.")